# Student Health Risk — Playground Series S6E7

**Task**: classificazione multi-classe (`fit` / `at-risk` / `unhealthy`)
**Metrica**: **balanced accuracy** → ogni classe pesa uguale, indipendentemente dal supporto
**Dataset**: 690.088 righe, 7 feature numeriche + 6 categoriche, null su tutte le feature

## Log delle versioni
| ver | descrizione | OOF BA (argmax) | OOF BA (prior-corr, t*) | LB |
|-----|-------------|-----------------|--------------------------|-----|
| v1  | LightGBM baseline, NaN crudi, prior-correction | 0.87626 | **0.94938** (t=1.00) | **0.94591** ✓ |
| v2  | v1 + `bmi_is_missing` (ablation) | = v1 | **delta 0.00000** → scartata | — |
| v3  | Optuna (obiettivo: BA post prior-correction) | 0.87824 | **0.94979** (t=1.00) | **0.94961** ✓ |
| v4  | CatBoost (GPU), parametri sobri | — | 0.94886 | — |
| v5  | Blend LGB+CB, w*(lgb)=0.95, t*=1.00 | — | 0.94981 (+0.00002) | 0.94960 |
| v6  | Optuna su CatBoost (GPU) | — | 0.94923 (t=0.95) → blend w*(lgb)=1.0 | — |
| v7  | FE interazioni: 6 candidate, 0 sopravvissute allo screening | — | = v3 | — |
| v8  | MLP + embedding (test ipotesi rumore) | — | *(in corso)* | — |
| v3b | Moltiplicatori liberi per classe su OOF v3 (zero retraining) | — | *(in corso)* | — |
| v9  | Sample weights bilanciati + correzione (l'ablation rimandata da S6E6) | — | *(in corso)* | — |

## Decisioni chiave (e perché)
1. **Nessuna imputation nella baseline**: LightGBM gestisce i NaN nativamente (impara la direzione dello split); le categoriche trattano il NaN come categoria propria. Qualsiasi imputation dovrà *battere* questa baseline in ablation.
2. **Prior-correction**: con sbilanciamento 86/8/6 e balanced accuracy, l'argmax del posteriore è subottimale. Si usa `argmax P(c|x) / prior^t`. Sweep su t: **t*=1.00** → probabilità ben calibrate (stesso risultato di S6E6).
3. **Diagnostica missingness** (vedi sotto): missingness ≈ MCAR ovunque tranne `bmi`, il cui tasso di null varia 4x tra `unhealthy` (0.71%) e `fit` (2.86%) → unico indicatore candidato come feature (v2).
4. **Co-missingness `exercise_duration`/`diet_type`**: entrambe 6.901 null, ma overlap osservato 72 ≈ 69 atteso sotto indipendenza → stessi conteggi marginali, nessun pattern strutturato.

**Ancoraggio OOF↔LB verificato** (v1): gap 0.0035, fisiologico (public LB = frazione del test; t* scelto su OOF → lieve ottimismo). OOF su 690k righe = stima più affidabile del public LB.

**Nota v3**: delta OOF +0.00040 (LightGBM spremuto: da qui in poi il margine si chiama diversita', non tuning). t*=1.00 per la terza volta consecutiva → probabilita' LightGBM ben calibrate, correzione bayesiana pura ottimale. Il LB (+0.0037) sovrastima il delta reale: rumore favorevole del public split, la bussola resta l'OOF.

**Post-mortem ensemble (v4/v5)**: accordo predizioni 99.29%, correlazione proba 0.9974, errori in comune 40.249 su ~42-43k. I due meccanismi (split vs ordered target statistics) convergono sugli stessi errori → forte indizio di **rumore irriducibile** sulle righe ambigue (lezione STAR↔GALAXY di S6E6). Il blend non ha fallito: ha *diagnosticato* l'assenza di diversita'.

**Post-mortem v6/v7**: il tuning ha migliorato CatBoost (+0.00035) rendendolo *piu' simile* a LightGBM (accordo 0.9929 → 0.9942): convergenza verso lo stesso confine bayesiano. Blend v3+v6: w*=1.0 (CatBoost escluso in toto). FE: tutti i delta entro il rumore di fold. **Plateau certificato per la famiglia alberi.** La v8 e' il test finale: una geometria non-ad-alberi sbaglia altrove? Accordo <0.98 → riapre il blend; ≥0.99 → sigillo definitivo sul rumore irriducibile.

**Lezioni dal notebook community "LB-guided logit ensemble" (LB 0.95088)**: (1) i loro "decision multipliers" (fit ×1.125, unhealthy ×1.273, tunati su OOF) sono la generalizzazione a 2 gradi di liberta' della nostra prior-correction → adottata come v3b; (2) il loro trio GBDT proprietario fa OOF 0.95008 e concorda al 99.5% coi file pubblici — conferma indipendente del plateau e del rumore irriducibile (mescolarlo nel blend PEGGIORAVA il LB di 0.00035: diluizione, non diversita'); (3) il loro 0.95088 viene dall'arbitraggio dei file pubblici (ensemble del lavoro della leaderboard), NON da modelli migliori — strada deliberatamente non percorsa qui; (4) i loro delta "LB-validated" (+0.0001) sono sotto il rumore del public split (±0.001): protocollo rigoroso, strumento di misura inadeguato; (5) media geometrica (log-space) > aritmetica quando i membri del blend hanno calibrazioni diverse → adottata nel confronto blend v3+v8.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

import kagglehub
kagglehub.competition_download('playground-series-s6e7')
BASE = '/kaggle/input/competitions/playground-series-s6e7'

SEED = 42
N_FOLDS = 5
print('Setup ✓')

In [ ]:
train = pd.read_csv(f'{BASE}/train.csv')
test  = pd.read_csv(f'{BASE}/test.csv')
print(train.shape, test.shape)

## 1. EDA essenziale

Target fortemente sbilanciato: `at-risk` 85.9%, `unhealthy` 8.4%, `fit` 5.8%.
Con la balanced accuracy, il campo di battaglia sono le due classi minoritarie.

In [ ]:
display(train['health_condition'].value_counts())
display(train['health_condition'].value_counts(normalize=True).round(4))
train.isnull().sum()

## 2. Diagnostica della missingness

Tre domande, in ordine:
1. i null di colonne con conteggi identici cadono sulle stesse righe? (co-missingness)
2. la missingness è correlata tra colonne? (pattern strutturati)
3. la missingness correla col **target**? (→ feature gratis)

**Esito** (prima esecuzione): missingness ≈ MCAR ovunque, unica eccezione `bmi`
(null: 0.71% in `unhealthy`, 2.08% in `at-risk`, 2.86% in `fit`).

In [ ]:
# 1. co-missingness: exercise_duration e diet_type hanno entrambe 6901 null
overlap = (train['exercise_duration'].isnull() & train['diet_type'].isnull()).sum()
expected = train['exercise_duration'].isnull().sum() * train['diet_type'].isnull().sum() / len(train)
print(f'overlap osservato: {overlap} | atteso sotto indipendenza: {expected:.0f}')

# 2. correlazione della missingness tra colonne
msno = train.drop(columns=['id', 'health_condition']).isnull()
plt.figure(figsize=(9, 7))
sns.heatmap(msno.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlazione della missingness')
plt.show()

# 3. frazione di null per classe (la domanda da un milione di punti)
null_by_class = train.groupby('health_condition')[msno.columns].apply(lambda g: g.isnull().mean()).T
display(null_by_class.round(4))

## 3. Preprocessing minimale

- NaN numerici: lasciati crudi (LightGBM li gestisce nativamente)
- Categoriche: cast a `category` (il NaN diventa di fatto una categoria)
- Target: LabelEncoder → **usare sempre `le.classes_` per rimappare le predizioni**, mai a mano

In [ ]:
TARGET = 'health_condition'
cat_cols = train.select_dtypes('object').columns.drop(TARGET).tolist()

le = LabelEncoder()
y = le.fit_transform(train[TARGET])
print('ordine classi:', dict(enumerate(le.classes_)))

def make_X(df, add_bmi_missing=False):
    X = df.drop(columns=['id', TARGET], errors='ignore').copy()
    if add_bmi_missing:
        X['bmi_is_missing'] = X['bmi'].isnull().astype(int)
    for c in cat_cols:
        X[c] = X[c].astype('category')
    return X

X      = make_X(train)
X_test = make_X(test)
priors = np.bincount(y) / len(y)
print('priors:', priors.round(4))

## 4. Pipeline CV riusabile

Un'unica funzione per baseline e ablation: stessi fold, stesso seed → confronti puliti.
Restituisce OOF, modelli di fold e BA per fold.

In [ ]:
LGB_PARAMS = dict(
    objective='multiclass', num_class=3,
    n_estimators=2000, learning_rate=0.05,
    num_leaves=63, random_state=SEED,
    verbosity=-1,
)

def run_cv(X, y, params=LGB_PARAMS, n_folds=N_FOLDS, seed=SEED, label=''):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = np.zeros((len(X), 3))
    models, fold_scores = [], []
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        model = lgb.LGBMClassifier(**params)
        model.fit(X.iloc[tr], y[tr],
                  eval_set=[(X.iloc[va], y[va])],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = model.predict_proba(X.iloc[va])
        models.append(model)
        ba = balanced_accuracy_score(y[va], oof[va].argmax(1))
        fold_scores.append(ba)
        print(f'[{label}] fold {fold}: BA = {ba:.5f}')
    print(f'[{label}] OOF BA (argmax): {balanced_accuracy_score(y, oof.argmax(1)):.5f}')
    return oof, models, fold_scores

def prior_sweep(oof, y, priors, ts=np.arange(0.0, 1.55, 0.25)):
    results = {}
    for t in ts:
        results[round(float(t), 2)] = balanced_accuracy_score(y, (oof / priors**t).argmax(1))
    for t, ba in results.items():
        print(f't={t:.2f}: BA = {ba:.5f}')
    t_star = max(results, key=results.get)
    print(f'--> t* = {t_star:.2f}, BA = {results[t_star]:.5f}')
    return t_star, results

## 5. v1 — baseline

Risultati prima esecuzione: fold BA 0.874–0.879 (CV stabilissima), OOF argmax **0.87626**,
prior-correction **+7.3 punti** → **0.94938** a t*=1.00.

In [ ]:
oof_v1, models_v1, _ = run_cv(X, y, label='v1')
t_star, sweep_v1 = prior_sweep(oof_v1, y, priors)

### Sweep fine intorno a t* (opzionale, per i perfezionisti)

In [ ]:
_ = prior_sweep(oof_v1, y, priors, ts=np.arange(0.85, 1.16, 0.05))

## 6. Submission (v1 + prior-correction)

Media delle `predict_proba` dei 5 modelli di fold, correzione con i prior del **train**, argmax,
rimappatura con `le.classes_`.

⚠️ Prima submission = **ancoraggio OOF ↔ LB**: se il LB ≈ 0.949, la macchina è verificata.

In [ ]:
T_SUB = 1.0  # aggiornare se lo sweep fine trova di meglio

test_proba = np.mean([m.predict_proba(X_test) for m in models_v1], axis=0)
test_pred  = le.classes_[(test_proba / priors**T_SUB).argmax(1)]

sub = pd.DataFrame({'id': test['id'], TARGET: test_pred})
sub.to_csv('submission.csv', index=False)
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()

## 7. v2 — ablation: `bmi_is_missing`

Ipotesi dalla diagnostica: il tasso di null di `bmi` varia 4x tra `unhealthy` e `fit`.
Stessi fold, stesso seed della v1 → il delta è attribuibile solo alla feature.

**Criterio di adozione**: delta OOF (dopo prior-correction) positivo e coerente sui fold.
Altrimenti si scarta, senza rimpianti.

**VERDETTO (eseguito)**: delta = 0.00000 spaccato. LightGBM sfrutta gia' il NaN di `bmi` negli split nativi; l'indicatore esplicito e' informazione ridondante. Feature scartata. Celle ricommentate per non sprecare compute nelle run successive.

In [ ]:
# X_v2      = make_X(train, add_bmi_missing=True)
# X_test_v2 = make_X(test,  add_bmi_missing=True)

# oof_v2, models_v2, _ = run_cv(X_v2, y, label='v2')
# t_star_v2, sweep_v2 = prior_sweep(oof_v2, y, priors)

# confronto onesto: delta a parita' di fold e seed, dopo prior-correction
# ba_v1 = max(sweep_v1.values())
# ba_v2 = max(sweep_v2.values())
# print(f'\nv1: {ba_v1:.5f} | v2: {ba_v2:.5f} | delta: {ba_v2 - ba_v1:+.5f}')

## 8. v3 — Optuna

Regole d'ingaggio:
1. **La funzione obiettivo ottimizza la BA *dopo* prior-correction** (sweep su t interno, costa solo argmax) — ottimizzare l'argmax nudo significherebbe ottimizzare la metrica sbagliata.
2. **Trial baseline in coda** (`enqueue_trial`) = paracadute anti-regressione: il best non potrà mai essere peggio della v1.
3. **Tuning a 3 fold** (vs 5 finali) + `MedianPruner` sui fold: con 690k righe il budget va speso con giudizio. I parametri vincenti vengono poi ri-validati con la pipeline standard a 5 fold, stessi fold della v1 → confronto pulito.

⏱️ ~30 trial × 3 fold su 690k righe: mettere in conto **qualche ora** su CPU Kaggle. Manopole: `N_TRIALS`, oppure `timeout=` in `study.optimize`.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_FOLDS = 3
N_TRIALS   = 30
T_GRID     = np.arange(0.85, 1.16, 0.05)

def ba_prior_corrected(y_true, proba, priors, ts=T_GRID):
    return max(balanced_accuracy_score(y_true, (proba / priors**t).argmax(1)) for t in ts)

def objective(trial):
    params = dict(
        objective='multiclass', num_class=3,
        n_estimators=3000, random_state=SEED, verbosity=-1, bagging_freq=1,
        learning_rate     = trial.suggest_float('learning_rate', 0.02, 0.10, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 31, 255, log=True),
        min_child_samples = trial.suggest_int('min_child_samples', 20, 300, log=True),
        feature_fraction  = trial.suggest_float('feature_fraction', 0.60, 1.00),
        bagging_fraction  = trial.suggest_float('bagging_fraction', 0.60, 1.00),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    )
    skf = StratifiedKFold(n_splits=TUNE_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros((len(X), 3))
    seen = np.zeros(len(X), dtype=bool)
    for i, (tr, va) in enumerate(skf.split(X, y)):
        m = lgb.LGBMClassifier(**params)
        m.fit(X.iloc[tr], y[tr],
              eval_set=[(X.iloc[va], y[va])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = m.predict_proba(X.iloc[va])
        seen[va] = True
        # BA parziale (sui fold visti finora) per il pruning
        trial.report(ba_prior_corrected(y[seen], oof[seen], priors), i)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return ba_prior_corrected(y, oof, priors)

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1),
)

# paracadute anti-regressione: la baseline v1 (con i default espliciti)
study.enqueue_trial(dict(
    learning_rate=0.05, num_leaves=63, min_child_samples=20,
    feature_fraction=1.0, bagging_fraction=1.0, reg_alpha=1e-3, reg_lambda=1e-3,
))

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'best trial #{study.best_trial.number}: BA (3-fold, post-corr) = {study.best_value:.5f}')
study.best_params

### Ri-validazione a 5 fold (stessi fold della v1) e confronto finale

Il punteggio a 3 fold e quello a 5 fold **non sono confrontabili tra loro**: il verdetto ufficiale
è sempre e solo la pipeline standard. Delta calcolato dopo prior-correction, a parità di fold.

In [ ]:
best_params = {**LGB_PARAMS, **study.best_params, 'n_estimators': 3000}

oof_v3, models_v3, _ = run_cv(X, y, params=best_params, label='v3')
t_star_v3, sweep_v3 = prior_sweep(oof_v3, y, priors, ts=np.arange(0.85, 1.16, 0.05))

ba_v1 = max(sweep_v1.values())
ba_v3 = max(sweep_v3.values())
print(f'\nv1: {ba_v1:.5f} | v3: {ba_v3:.5f} | delta: {ba_v3 - ba_v1:+.5f}')

### Submission v3

Solo se il delta è positivo e sensato; altrimenti la v1 resta la submission di riferimento
(e sarà il segnale che il margine va cercato altrove: ensemble con CatBoost).

In [ ]:
test_proba_v3 = np.mean([m.predict_proba(X_test) for m in models_v3], axis=0)
test_pred_v3  = le.classes_[(test_proba_v3 / priors**t_star_v3).argmax(1)]

sub = pd.DataFrame({'id': test['id'], TARGET: test_pred_v3})
sub.to_csv('submission.csv', index=False)
print(f't usato: {t_star_v3:.2f}')
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()

## 9. v4 — CatBoost (GPU)

Perché CatBoost e non "ancora tuning": il margine residuo è nella **diversità**, non nell'ottimizzazione.
CatBoost gestisce le categoriche con **ordered target statistics** (encoding basato sul target, calcolato
su permutazioni per evitare leakage) — un meccanismo genuinamente diverso dagli split su categoria di
LightGBM. Errori decorrelati = carburante per il blend.

Due differenze operative da non dimenticare:
1. **CatBoost NON accetta NaN nelle categoriche** (ironico, per un gatto): vanno riempiti con una
   categoria esplicita `'missing'`. I NaN numerici invece li gestisce nativamente, come LightGBM.
2. `task_type='GPU'`: supporto maturo, 690k righe diventano una passeggiata. È qui che si spendono
   le ore GPU risparmiate sul tuning LightGBM.

Pipeline identica alla v1/v3: stessi 5 fold, stesso seed, prior-correction in coda.

In [ ]:
from catboost import CatBoostClassifier

def make_X_cb(df):
    X_ = df.drop(columns=['id', TARGET], errors='ignore').copy()
    for c in cat_cols:
        X_[c] = X_[c].astype(object).fillna('missing').astype(str)
    return X_

X_cb      = make_X_cb(train)
X_test_cb = make_X_cb(test)

CB_PARAMS = dict(
    loss_function='MultiClass',
    iterations=3000, learning_rate=0.05, depth=8,
    task_type='GPU', devices='0',
    random_seed=SEED, verbose=0,
)

def run_cv_cb(X_, y, params=CB_PARAMS, n_folds=N_FOLDS, seed=SEED, label='cb'):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = np.zeros((len(X_), 3))
    models, fold_scores = [], []
    for fold, (tr, va) in enumerate(skf.split(X_, y)):
        model = CatBoostClassifier(**params, cat_features=cat_cols)
        model.fit(X_.iloc[tr], y[tr],
                  eval_set=(X_.iloc[va], y[va]),
                  early_stopping_rounds=100, use_best_model=True)
        oof[va] = model.predict_proba(X_.iloc[va])
        models.append(model)
        ba = balanced_accuracy_score(y[va], oof[va].argmax(1))
        fold_scores.append(ba)
        print(f'[{label}] fold {fold}: BA = {ba:.5f}')
    print(f'[{label}] OOF BA (argmax): {balanced_accuracy_score(y, oof.argmax(1)):.5f}')
    return oof, models, fold_scores

oof_v4, models_v4, _ = run_cv_cb(X_cb, y, label='v4')
t_star_v4, sweep_v4 = prior_sweep(oof_v4, y, priors, ts=np.arange(0.85, 1.16, 0.05))

ba_v3 = max(sweep_v3.values())
ba_v4 = max(sweep_v4.values())
print(f'\nv3 (lgb): {ba_v3:.5f} | v4 (cb): {ba_v4:.5f} | delta: {ba_v4 - ba_v3:+.5f}')

### Diagnostica di diversità

Prima ancora del punteggio solo-CatBoost, la domanda che decide il destino del blend:
**quanto sono decorrelati gli errori dei due modelli?**
Se sbagliano sulle stesse righe, il blend non ha nulla da guadagnare — anche con un ottimo punteggio v4.
Un CatBoost leggermente peggiore ma *diverso* vale più di un clone.

In [ ]:
pred_lgb = (oof_v3 / priors**1.0).argmax(1)
pred_cb  = (oof_v4 / priors**t_star_v4).argmax(1)

err_lgb = pred_lgb != y
err_cb  = pred_cb  != y
both    = (err_lgb & err_cb).sum()
only_l  = (err_lgb & ~err_cb).sum()
only_c  = (~err_lgb & err_cb).sum()

print(f'errori lgb: {err_lgb.sum()} | errori cb: {err_cb.sum()}')
print(f'in comune: {both} | solo lgb: {only_l} | solo cb: {only_c}')
print(f'accordo predizioni: {(pred_lgb == pred_cb).mean():.4f}')
print(f'correlazione proba (media per classe): '
      f'{np.mean([np.corrcoef(oof_v3[:,k], oof_v4[:,k])[0,1] for k in range(3)]):.4f}')

**VERDETTO (eseguito)**: errori lgb 42.074, cb 43.105, **in comune 40.249**; accordo 99.29%,
correlazione proba 0.9974. Tetto teorico del blend: anche recuperando *tutti* i 1.825 errori
esclusivi di lgb senza introdurre i 2.856 di cb, il guadagno resterebbe di pochi decimillesimi —
e infatti w*=0.95, delta +0.00002. Capitolo ensemble chiuso con i numeri.

## 10. v5 — Blend

Media pesata delle probabilità OOF: `w * lgb + (1-w) * cb`, con sweep congiunto di **w** e **t**
sull'OOF. Due soli iperparametri stimati su 690k righe di OOF: rischio di overfit trascurabile.

**Criterio**: il blend si adotta solo se batte il migliore dei due singoli.

In [ ]:
best = (-1, None, None)
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * oof_v3 + (1 - w) * oof_v4
    for t in np.arange(0.85, 1.16, 0.05):
        ba = balanced_accuracy_score(y, (blend / priors**t).argmax(1))
        if ba > best[0]:
            best = (ba, round(float(w), 2), round(float(t), 2))

ba_v5, w_star, t_star_v5 = best
print(f'v5 blend: BA = {ba_v5:.5f} con w(lgb) = {w_star}, t = {t_star_v5}')
print(f'vs migliore singolo: {max(ba_v3, ba_v4):.5f} | delta: {ba_v5 - max(ba_v3, ba_v4):+.5f}')

### Submission v5 (o fallback)

Se il blend vince: media pesata delle probabilità di test dei due set di modelli, poi correzione con
(w*, t*). Se non vince, la submission resta quella del migliore singolo — nessun affezionamento.

In [ ]:
test_proba_lgb = np.mean([m.predict_proba(X_test)    for m in models_v3], axis=0)
test_proba_cb  = np.mean([m.predict_proba(X_test_cb) for m in models_v4], axis=0)

test_blend = w_star * test_proba_lgb + (1 - w_star) * test_proba_cb
test_pred  = le.classes_[(test_blend / priors**t_star_v5).argmax(1)]

sub = pd.DataFrame({'id': test['id'], TARGET: test_pred})
sub.to_csv('submission.csv', index=False)
print(f'blend: w(lgb)={w_star}, t={t_star_v5}')
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()

## 11. v6 — Optuna su CatBoost (GPU)

Confronto v3 vs v4 era ingiusto (30 trial vs parametri sobri): qui si pareggia il campo.
Aspettative dichiarate: anche un CatBoost tunato difficilmente aggiungerà diversità (la diagnostica
sopra vale più del tuning), ma l'esperimento certifica il plateau *oppure* ci smentisce — win-win
epistemico, e il costo lo paga la GPU.

Note GPU-specifiche:
- `rsm` (colsample) **non supportato** in multiclass su GPU → si regolarizza con `depth`,
  `l2_leaf_reg`, `min_data_in_leaf`, `random_strength`, `bagging_temperature` (bootstrap bayesiano)
- stesso protocollo della v3: obiettivo = BA **post prior-correction**, 3 fold, subsample
  stratificato 35%, `enqueue_trial` della v4 come paracadute

In [ ]:
from sklearn.model_selection import train_test_split

idx_tune, _ = train_test_split(np.arange(len(X_cb)), train_size=0.35,
                               stratify=y, random_state=SEED)
X_tune_cb, y_tune = X_cb.iloc[idx_tune].reset_index(drop=True), y[idx_tune]
priors_tune = np.bincount(y_tune) / len(y_tune)

def objective_cb(trial):
    params = dict(
        loss_function='MultiClass', iterations=3000,
        task_type='GPU', devices='0', random_seed=SEED, verbose=0,
        bootstrap_type='Bayesian',
        learning_rate       = trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        depth               = trial.suggest_int('depth', 4, 10),
        l2_leaf_reg         = trial.suggest_float('l2_leaf_reg', 1.0, 30.0, log=True),
        min_data_in_leaf    = trial.suggest_int('min_data_in_leaf', 1, 300, log=True),
        random_strength     = trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 2.0),
    )
    skf = StratifiedKFold(n_splits=TUNE_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros((len(X_tune_cb), 3))
    seen = np.zeros(len(X_tune_cb), dtype=bool)
    for i, (tr, va) in enumerate(skf.split(X_tune_cb, y_tune)):
        m = CatBoostClassifier(**params, cat_features=cat_cols)
        m.fit(X_tune_cb.iloc[tr], y_tune[tr],
              eval_set=(X_tune_cb.iloc[va], y_tune[va]),
              early_stopping_rounds=100, use_best_model=True)
        oof[va] = m.predict_proba(X_tune_cb.iloc[va])
        seen[va] = True
        trial.report(ba_prior_corrected(y_tune[seen], oof[seen], priors_tune), i)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return ba_prior_corrected(y_tune, oof, priors_tune)

study_cb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1),
)
# paracadute: la v4
study_cb.enqueue_trial(dict(learning_rate=0.05, depth=8, l2_leaf_reg=3.0,
                            min_data_in_leaf=1, random_strength=1.0, bagging_temperature=1.0))
study_cb.optimize(objective_cb, n_trials=25, show_progress_bar=True)

print(f'best: BA (3-fold/35%, post-corr) = {study_cb.best_value:.5f}')
study_cb.best_params

In [ ]:
# ri-validazione ufficiale: 5 fold, dataset pieno, stessi fold di sempre
CB_BEST = {**CB_PARAMS, **study_cb.best_params, 'bootstrap_type': 'Bayesian'}

oof_v6, models_v6, _ = run_cv_cb(X_cb, y, params=CB_BEST, label='v6')
t_star_v6, sweep_v6 = prior_sweep(oof_v6, y, priors, ts=np.arange(0.85, 1.16, 0.05))

ba_v6 = max(sweep_v6.values())
print(f'\nv4 (cb sobrio): {max(sweep_v4.values()):.5f} | v6 (cb tunato): {ba_v6:.5f}')
print(f'v3 (lgb): {ba_v3:.5f} | delta v6-v3: {ba_v6 - ba_v3:+.5f}')

# ri-diagnosi diversita' e ri-blend con il gatto tunato
pred_cb6 = (oof_v6 / priors**t_star_v6).argmax(1)
print(f'accordo lgb-cb6: {(pred_lgb == pred_cb6).mean():.4f}')

best = (-1, None, None)
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * oof_v3 + (1 - w) * oof_v6
    for t in np.arange(0.85, 1.16, 0.05):
        ba = balanced_accuracy_score(y, (blend / priors**t).argmax(1))
        if ba > best[0]:
            best = (ba, round(float(w), 2), round(float(t), 2))
print(f'blend v3+v6: BA = {best[0]:.5f} con w(lgb) = {best[1]}, t = {best[2]}')

## 12. v7 — Feature engineering: interazioni

L'unica via che può **aggiungere informazione** invece di rimescolarla. Candidate: rapporti e
interazioni fisiologicamente sensate tra le 7 numeriche. Nota tecnica a favore dei rapporti:
la divisione propaga i NaN (NaN in uno dei due → NaN nel rapporto), che LightGBM gestisce
nativamente — nessuna imputation richiesta nemmeno qui.

Protocollo: **screening a 3 fold** (una feature alla volta, delta post-correction vs baseline
3-fold), poi le sopravvissute — se esistono — si ri-validano insieme a 5 fold contro la v3.
Gli alberi *possono* costruirsi le interazioni da soli con abbastanza split, ma dargliele
esplicite abbassa il costo in profondità: se il segnale c'è, il delta si vede.

In [ ]:
FE_CANDIDATES = {
    'cal_per_step':    lambda X_: X_['calorie_expenditure'] / X_['step_count'],
    'cal_per_ex_min':  lambda X_: X_['calorie_expenditure'] / X_['exercise_duration'],
    'water_per_cal':   lambda X_: X_['water_intake'] / X_['calorie_expenditure'],
    'activity_load':   lambda X_: X_['step_count'] * X_['exercise_duration'],
    'hr_x_bmi':        lambda X_: X_['heart_rate'] * X_['bmi'],
    'sleep_x_hr':      lambda X_: X_['sleep_duration'] * X_['heart_rate'],
}

def make_X_fe(df, features=()):
    X_ = make_X(df)
    for name in features:
        X_[name] = FE_CANDIDATES[name](X_)
    return X_

def quick_ba(X_, y, n_folds=3):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    oof = np.zeros((len(X_), 3))
    for tr, va in skf.split(X_, y):
        m = lgb.LGBMClassifier(**best_params)
        m.fit(X_.iloc[tr], y[tr], eval_set=[(X_.iloc[va], y[va])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = m.predict_proba(X_.iloc[va])
    return ba_prior_corrected(y, oof, priors)

ba_ref = quick_ba(make_X_fe(train), y)
print(f'baseline 3-fold: {ba_ref:.5f}\n')

results_fe = {}
for name in FE_CANDIDATES:
    ba = quick_ba(make_X_fe(train, [name]), y)
    results_fe[name] = ba - ba_ref
    print(f'{name:16s}: BA = {ba:.5f} | delta = {ba - ba_ref:+.5f}')

In [ ]:
# ri-validazione a 5 fold delle sole sopravvissute (delta screening > +0.0005)
SURVIVORS = [k for k, v in results_fe.items() if v > 0.0005]
print('sopravvissute allo screening:', SURVIVORS or 'nessuna')

if SURVIVORS:
    X_v7      = make_X_fe(train, SURVIVORS)
    X_test_v7 = make_X_fe(test,  SURVIVORS)
    oof_v7, models_v7, _ = run_cv(X_v7, y, params=best_params, label='v7')
    t_star_v7, sweep_v7 = prior_sweep(oof_v7, y, priors, ts=np.arange(0.85, 1.16, 0.05))
    print(f'\nv3: {ba_v3:.5f} | v7: {max(sweep_v7.values()):.5f} | '
          f'delta: {max(sweep_v7.values()) - ba_v3:+.5f}')

### Criterio di chiusura

- Se v6 e v7 escono entrambe a delta ≈ 0: **plateau certificato**. Submission finale = v3
  (scelta su OOF, non su public LB), post-mortem nel log versioni, competizione consolidata.
- Se qualcosa sopravvive: si integra, si ri-ancora al LB, e si ripete la domanda.

## 13. v8 — MLP con embedding (GPU)

**Obiettivo dichiarato**: NON battere LightGBM (improbabile), ma rispondere all'ultima domanda aperta —
*una geometria non-ad-alberi (confini lisci vs partizioni a scalini) sbaglia su righe diverse?*
Il numero che conta a fine run non è la BA: è l'**accordo con lgb**.

Scelte di design, e i perché:
- **Imputation: mediana + flag di missingness**, obbligatoria qui (NaN × peso = NaN). Con missingness
  ≈ MCAR l'imputation semplice è quasi-ottimale: non c'è segnale nei buchi da ricostruire, serve solo
  un valore innocuo. I flag restituiscono alla rete l'informazione "qui c'era un buco".
- **Statistiche per fold, dal solo train**: mediana, media e deviazione standard fittate sul train
  fold — una mediana fittata su train+val è leakage in miniatura, ma sempre leakage.
- **Embedding per le categoriche** (NaN = categoria 'missing' anche qui, coerenti con la filosofia).
- **Training con log-loss naturale + prior-correction in coda**: stessa pipeline di tutto il resto —
  la CrossEntropy produce probabilità calibrate, la correzione fa il suo lavoro dopo.
- Stessi 5 fold, stesso seed: i confronti restano puliti.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

num_cols = [c for c in X.columns if c not in cat_cols]

# vocabolari categorici globali (solo vocabolario, nessun target: non e' leakage)
cat_maps = {c: {v: i + 1 for i, v in enumerate(sorted(train[c].dropna().unique()))}
            for c in cat_cols}                     # 0 = missing
cat_cards = [len(cat_maps[c]) + 1 for c in cat_cols]

def cat_codes(df):
    out = np.zeros((len(df), len(cat_cols)), dtype=np.int64)
    for j, c in enumerate(cat_cols):
        out[:, j] = df[c].map(cat_maps[c]).fillna(0).astype(np.int64)
    return out

def num_block(df, medians, mu, sd):
    vals  = df[num_cols].fillna(medians)
    flags = df[num_cols].isnull().astype(np.float32)
    z     = ((vals - mu) / sd).astype(np.float32)
    return np.hstack([z.values, flags.values])

CAT_TRAIN, CAT_TEST = cat_codes(train), cat_codes(test)

In [ ]:
class TabMLP(nn.Module):
    def __init__(self, cat_cards, n_num, n_classes=3):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(card, min(8, (card + 2) // 2))
                                   for card in cat_cards])
        d_in = sum(e.embedding_dim for e in self.embs) + n_num
        self.net = nn.Sequential(
            nn.Linear(d_in, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.25),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.25),
            nn.Linear(128, n_classes),
        )
    def forward(self, x_num, x_cat):
        e = [emb(x_cat[:, j]) for j, emb in enumerate(self.embs)]
        return self.net(torch.cat(e + [x_num], dim=1))

def train_fold(Xn_tr, Xc_tr, y_tr, Xn_va, Xc_va, y_va,
               epochs=30, patience=3, bs=8192, lr=2e-3):
    tr_dl = DataLoader(TensorDataset(torch.tensor(Xn_tr), torch.tensor(Xc_tr),
                                     torch.tensor(y_tr)), batch_size=bs, shuffle=True)
    Xn_va_t = torch.tensor(Xn_va).to(device)
    Xc_va_t = torch.tensor(Xc_va).to(device)
    y_va_t  = torch.tensor(y_va).to(device)

    model = TabMLP(cat_cards, Xn_tr.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = nn.CrossEntropyLoss()

    best_loss, best_state, bad = np.inf, None, 0
    for ep in range(epochs):
        model.train()
        for xn, xc, yb in tr_dl:
            xn, xc, yb = xn.to(device), xc.to(device), yb.to(device)
            opt.zero_grad()
            loss = lossf(model(xn, xc), yb)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            va_loss = lossf(model(Xn_va_t, Xc_va_t), y_va_t).item()
        if va_loss < best_loss - 1e-5:
            best_loss, bad = va_loss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model, best_loss

In [ ]:
torch.manual_seed(SEED)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_v8 = np.zeros((len(train), 3))
test_proba_v8 = np.zeros((len(test), 3))

for fold, (tr, va) in enumerate(skf.split(X, y)):
    med = train.iloc[tr][num_cols].median()
    mu  = train.iloc[tr][num_cols].fillna(med).mean()
    sd  = train.iloc[tr][num_cols].fillna(med).std() + 1e-8

    Xn_tr = num_block(train.iloc[tr], med, mu, sd)
    Xn_va = num_block(train.iloc[va], med, mu, sd)
    Xn_te = num_block(test,           med, mu, sd)

    model, vloss = train_fold(Xn_tr, CAT_TRAIN[tr], y[tr], Xn_va, CAT_TRAIN[va], y[va])

    with torch.no_grad():
        oof_v8[va] = torch.softmax(model(torch.tensor(Xn_va).to(device),
                                         torch.tensor(CAT_TRAIN[va]).to(device)), 1).cpu().numpy()
        test_proba_v8 += torch.softmax(model(torch.tensor(Xn_te).to(device),
                                             torch.tensor(CAT_TEST).to(device)), 1).cpu().numpy() / N_FOLDS
    ba = balanced_accuracy_score(y[va], oof_v8[va].argmax(1))
    print(f'[v8] fold {fold}: BA = {ba:.5f} (val loss {vloss:.5f})')

print(f'[v8] OOF BA (argmax): {balanced_accuracy_score(y, oof_v8.argmax(1)):.5f}')
t_star_v8, sweep_v8 = prior_sweep(oof_v8, y, priors, ts=np.arange(0.85, 1.16, 0.05))

### Il numero che conta: accordo e blend

- accordo lgb-mlp **< 0.98** → geometria diversa = errori diversi: il blend ha materia prima, si riapre il capitolo
- accordo **≥ 0.99** → anche una geometria radicalmente diversa converge sugli stessi errori:
  **sigillo definitivo** sul rumore irriducibile, si consolida con la coscienza pulita dello scienziato

In [ ]:
pred_mlp = (oof_v8 / priors**t_star_v8).argmax(1)
err_mlp = pred_mlp != y

print(f'errori lgb: {err_lgb.sum()} | errori mlp: {err_mlp.sum()}')
print(f'in comune: {(err_lgb & err_mlp).sum()} | solo lgb: {(err_lgb & ~err_mlp).sum()} '
      f'| solo mlp: {(~err_lgb & err_mlp).sum()}')
print(f'accordo predizioni: {(pred_lgb == pred_mlp).mean():.4f}')
print(f'correlazione proba: '
      f'{np.mean([np.corrcoef(oof_v3[:,k], oof_v8[:,k])[0,1] for k in range(3)]):.4f}')

# blend: media aritmetica VS geometrica (log-space, robusta a calibrazioni diverse)
EPS = 1e-7
log_lgb, log_mlp = np.log(np.clip(oof_v3, EPS, 1)), np.log(np.clip(oof_v8, EPS, 1))

def sweep_blend(mix_fn, name):
    best = (-1, None, None)
    for w in np.arange(0.0, 1.01, 0.05):
        blend = mix_fn(w)
        for t in np.arange(0.85, 1.16, 0.05):
            ba = balanced_accuracy_score(y, (blend / priors**t).argmax(1))
            if ba > best[0]:
                best = (ba, round(float(w), 2), round(float(t), 2))
    print(f'{name:12s}: BA = {best[0]:.5f} con w(lgb) = {best[1]}, t = {best[2]} '
          f'(vs v3: {best[0] - ba_v3:+.5f})')
    return best

best_ari = sweep_blend(lambda w: w * oof_v3 + (1 - w) * oof_v8, 'aritmetica')
best_geo = sweep_blend(lambda w: np.exp(w * log_lgb + (1 - w) * log_mlp), 'geometrica')

## 14. v3b — Moltiplicatori liberi per classe (zero retraining)

Generalizzazione della prior-correction, rubata (con attribuzione) al notebook community:
da **argmax p/prior^t** (1 grado di libertà) ad **argmax p·m** con m = (m_fit, 1, m_unhealthy)
liberi (2 gradi di libertà, maggioranza fissa a 1). Con 690k righe OOF, 2 parametri non fanno
overfitting. Partenza dalla correzione bayesiana pura (p/prior), raffinamento coarse→fine.

Aspettativa dichiarata: +0.0000/+0.0002 — t*=1.00 costantemente ottimale suggerisce poco residuo
da correggere. Ma costa trenta secondi di argmax.

In [ ]:
from itertools import product

IDX_FIT = int(le.transform(['fit'])[0])
IDX_UNH = int(le.transform(['unhealthy'])[0])
base = oof_v3 / priors            # correzione bayesiana pura (t=1) come punto di partenza

def mult_search(grid_f, grid_u):
    best = (-1, None, None)
    for mf, mu in product(grid_f, grid_u):
        m = np.ones(3); m[IDX_FIT] = mf; m[IDX_UNH] = mu
        ba = balanced_accuracy_score(y, (base * m).argmax(1))
        if ba > best[0]:
            best = (ba, mf, mu)
    return best

# coarse -> fine
ba_c, mf_c, mu_c = mult_search(np.linspace(0.7, 1.6, 19), np.linspace(0.7, 1.6, 19))
ba_f, mf_f, mu_f = mult_search(np.linspace(mf_c - 0.05, mf_c + 0.05, 11),
                               np.linspace(mu_c - 0.05, mu_c + 0.05, 11))

print(f'v3  (t-sweep):        BA = {ba_v3:.5f}')
print(f'v3b (moltiplicatori): BA = {ba_f:.5f} con m_fit={mf_f:.3f}, m_unhealthy={mu_f:.3f}')
print(f'delta: {ba_f - ba_v3:+.5f}')

## 15. v9 — Sample weights bilanciati + correzione

**L'ablation rimandata da S6E6**, finalmente. Domanda: addestrare con pesi bilanciati
(`class_weight='balanced'`) e poi correggere la decisione batte l'addestramento naturale + correzione?

Teoria: i pesi bilanciati spostano la loss *durante* il training (il modello vede le minoritarie
"ingrandite" 15x → confini diversi, probabilità NON più calibrate sui prior naturali); la
prior-correction sposta la *decisione* a training invariato. Il notebook community usa la prima
strada (+ moltiplicatori residui); noi finora la seconda. Stessi fold, stesso seed, e in coda a
v9 si applica la ricerca dei moltiplicatori liberi (il t-sweep non basta: con pesi bilanciati i
prior "interni" del modello non sono più quelli naturali, la correzione residua va stimata libera).

In [ ]:
params_v9 = {**best_params, 'class_weight': 'balanced'}
oof_v9, models_v9, _ = run_cv(X, y, params=params_v9, label='v9')

# con pesi bilanciati la correzione residua va cercata libera (non solo t-sweep)
base9 = oof_v9 / priors
def mult_search_on(base_):
    best = (-1, None, None)
    for mf, mu in product(np.linspace(0.3, 1.8, 31), np.linspace(0.3, 1.8, 31)):
        m = np.ones(3); m[IDX_FIT] = mf; m[IDX_UNH] = mu
        ba = balanced_accuracy_score(y, (base_ * m).argmax(1))
        if ba > best[0]:
            best = (ba, mf, mu)
    return best

ba9_raw = balanced_accuracy_score(y, oof_v9.argmax(1))
_, s9 = None, prior_sweep(oof_v9, y, priors, ts=np.arange(0.0, 1.16, 0.05))[1]
ba9_mult, mf9, mu9 = mult_search_on(base9)

print(f'\nv9 argmax nudo:        {ba9_raw:.5f}  (nota: coi pesi bilanciati parte gia\' alto)')
print(f'v9 + t-sweep:          {max(s9.values()):.5f}')
print(f'v9 + moltiplicatori:   {ba9_mult:.5f} (m_fit={mf9:.2f}, m_unhealthy={mu9:.2f})')
print(f'\nvs v3/v3b (naturale + correzione): {max(ba_v3, ba_f):.5f} | '
      f'delta: {ba9_mult - max(ba_v3, ba_f):+.5f}')

## 16. Chiusura

Con v8 (geometria), v3b (decisione), v9 (loss) esplorate, le tre leve fondamentali —
*rappresentazione, regola di decisione, funzione obiettivo* — sono tutte state testate.
Qualunque sia l'esito, la submission finale si sceglie **sull'OOF**, il log versioni si completa,
e il post-mortem certifica dove finiva il segnale e dove iniziava il rumore del generatore.